# DDRI 군집별 모델링 템플릿(Cluster Modeling Template)

이 노트북은 `works/05_prediction_long/05_ddri_team_cluster_modeling_protocol.md` 기준으로, 대표 대여소 15개 중 특정 `station_group(담당 군집 그룹명)` 1개를 맡아 공통 실험을 수행하기 위한 템플릿이다.

사용 방법:

1. 이 파일을 복사한다.
2. 파일명을 담당 군집에 맞게 바꾼다.
3. `TARGET_STATION_GROUP(대상 군집 그룹명)` 값을 본인 군집명으로 바꾼다.
4. 아래 순서를 그대로 따라 실험한다.


## 용어 설명
- `feature(입력 변수)`: 모델이 예측에 사용하는 입력 값
- `lag(과거 시점 값)`: 같은 대여소의 몇 시간 전 대여량을 가져온 피처
- `rolling(이동 통계)`: 최근 몇 시간 대여량의 평균이나 표준편차를 계산한 피처
- `Train / Validation / Test(학습 / 검증 / 최종 평가)`: 모델을 학습하고 비교하고 마지막으로 점검하는 데이터 구간
- `objective(학습 목표 함수)`: 모델이 어떤 방식으로 오차를 줄이도록 학습할지 정하는 기준
- `RMSE(제곱평균제곱근오차)`: 큰 오차에 더 민감한 대표 회귀 평가 지표
- `MAE(평균절대오차)`: 실제값과 예측값의 절대 차이 평균
- `R²(설명력)`: 모델이 변동을 얼마나 설명하는지 보는 지표


## 1. 실험 목적(Experiment Goal, 실험 목표)

- 담당 군집: `여기에 군집명 기입`
- 목표: 대표 대여소 3개를 대상으로 `2023 train(학습) / 2024 validation(검증) / 2025 test(최종 평가)` 정책에 따라 공통 모델 실험을 수행한다.
- 필수 모델: `LinearRegression(선형회귀 모델)`, `LightGBM_RMSE(RMSE 기준 LightGBM)`
- 필수 평가 지표: `RMSE(제곱평균제곱근오차)`, `MAE(평균절대오차)`, `R²(설명력)`


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long')
TRAIN_PATH = Path('/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/raw_data/ddri_prediction_long_train_2023_2024.csv')
TEST_PATH = Path('/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/raw_data/ddri_prediction_long_test_2025.csv')
OUTPUT_DIR = BASE_DIR / 'output' / 'team_cluster_template_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_STATION_GROUP = '아침 도착 업무 집중형'  # 담당 군집 고정
DISPLAY_NAME = TARGET_STATION_GROUP

RANDOM_STATE = 42


## 2. 입력 데이터 로드(Input Data Load, 입력 데이터 불러오기)

- 공통 입력 정본은 `3조 공유폴더/대표대여소_예측데이터_15개/raw_data/` 아래 CSV 2개다.
- 개인이 별도 가공한 CSV를 정본처럼 사용하지 않는다.


In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('train shape:', train_raw.shape)
print('test shape:', test_raw.shape)
print('station groups:', sorted(train_raw['station_group'].dropna().unique().tolist()))


train shape: (263160, 15)
test shape: (131400, 15)
station groups: ['생활권 혼합형', '아침 도착 업무 집중형', '업무/상업 혼합형', '외곽 주거형', '주거 도착형']


## 3. 담당 군집 필터링(Filter Assigned Cluster, 담당 군집 추리기)

- 반드시 `station_group(담당 군집 그룹명)` 1개만 남긴다.
- 결과적으로 대표 대여소 3개만 남아야 한다.


In [3]:
train_group = train_raw.loc[train_raw['station_group'] == TARGET_STATION_GROUP].copy()
test_group = test_raw.loc[test_raw['station_group'] == TARGET_STATION_GROUP].copy()

print('train_group shape:', train_group.shape)
print('test_group shape:', test_group.shape)
print(train_group[['station_id', 'station_name']].drop_duplicates().sort_values('station_id').to_string(index=False))


train_group shape: (52632, 15)
test_group shape: (26280, 15)
 station_id   station_name
       2323  주식회사 오뚜기 정문 앞
       2348   포스코사거리(기업은행)
       2377       수서역 5번출구


## 4. 전처리 및 파생 피처 생성(Preprocessing And Feature Engineering, 전처리와 파생 피처 생성)

공통 프로토콜:

- `date(날짜)`는 날짜형으로 변환한다.
- `station_id(숫자 대여소 ID)`, `date(날짜)`, `hour(시간)` 기준으로 정렬한다.
- `lag_1h(1시간 전 대여량)`, `lag_24h(24시간 전 대여량)`, `lag_168h(168시간 전 대여량)`를 만든다.
- `rolling_mean_24h(최근 24시간 대여량 이동평균)`, `rolling_std_24h(최근 24시간 대여량 이동표준편차)`, `rolling_mean_168h(최근 168시간 대여량 이동평균)`, `rolling_std_168h(최근 168시간 대여량 이동표준편차)`를 만든다.
- rolling(이동 통계)은 반드시 `station_id(숫자 대여소 ID)`별로 계산하고, `shift(1)(직전 시점으로 한 칸 미루기)` 후 rolling을 적용한다.
- `lag_168h(168시간 전 대여량)`는 `1주 전 동일 요일·동일 시각 대여량`이다.


In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['date'] = pd.to_datetime(result['date'])
    result = result.sort_values(['station_id', 'date', 'hour']).reset_index(drop=True)

    group = result.groupby('station_id', group_keys=False)['rental_count']
    result['lag_1h'] = group.shift(1)
    result['lag_24h'] = group.shift(24)
    result['lag_168h'] = group.shift(168)

    shifted = group.shift(1)
    result['rolling_mean_24h'] = shifted.rolling(24).mean().reset_index(level=0, drop=True)
    result['rolling_std_24h'] = shifted.rolling(24).std().reset_index(level=0, drop=True)
    result['rolling_mean_168h'] = shifted.rolling(168).mean().reset_index(level=0, drop=True)
    result['rolling_std_168h'] = shifted.rolling(168).std().reset_index(level=0, drop=True)
    return result

train_feat = build_features(train_group)
test_feat = build_features(test_group)

train_feat[['station_id', 'date', 'hour', 'rental_count', 'lag_1h', 'lag_24h', 'lag_168h']].head()


,station_id,date,hour,rental_count,lag_1h,lag_24h,lag_168h
0,2323,2023-01-01,0,0,NaN,NaN,NaN
1,2323,2023-01-01,1,0,0.0,NaN,NaN
2,2323,2023-01-01,2,0,0.0,NaN,NaN
3,2323,2023-01-01,3,0,0.0,NaN,NaN
4,2323,2023-01-01,4,0,0.0,NaN,NaN


## 5. 시간 분할 정책(Time-Based Split Policy, 시간 기준 분할 정책)

- 학습(`Train`, 학습 구간): `2023`
- 검증(`Validation`, 검증 구간): `2024`
- 테스트(`Test`, 최종 평가 구간): `2025`

랜덤 분할은 금지한다.


In [5]:
train_2023 = train_feat.loc[train_feat['date'].dt.year == 2023].copy()
valid_2024 = train_feat.loc[train_feat['date'].dt.year == 2024].copy()
test_2025 = test_feat.copy()

print('train_2023:', train_2023.shape)
print('valid_2024:', valid_2024.shape)
print('test_2025:', test_2025.shape)


train_2023: (26280, 22)
valid_2024: (26352, 22)
test_2025: (26280, 22)


## 6. 입력 피처 목록(Input Feature List, 입력 피처 목록)

아래 목록은 공통 기본 피처(feature, 입력 변수)다. 개인 실험에서 피처를 추가하거나 제거했다면 반드시 이유를 마크다운에 남긴다.


In [6]:
target_col = 'rental_count'

categorical_features = [
    'station_id',
    'cluster',
    'mapped_dong_code',
]

numeric_features = [
    'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h',
    'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h',
]

feature_cols = categorical_features + numeric_features
feature_cols


['station_id',
 'cluster',
 'mapped_dong_code',
 'hour',
 'weekday',
 'month',
 'holiday',
 'temperature',
 'humidity',
 'precipitation',
 'wind_speed',
 'lag_1h',
 'lag_24h',
 'lag_168h',
 'rolling_mean_24h',
 'rolling_std_24h',
 'rolling_mean_168h',
 'rolling_std_168h']

## 7. 평가 함수(Evaluation Function, 평가 함수)

모든 팀원은 아래 3개 점수를 동일하게 기록한다.

- `RMSE(제곱평균제곱근오차)`
- `MAE(평균절대오차)`
- `R²(설명력)`


In [7]:
def evaluate_regression(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def build_linear_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ]
    )
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ])

def build_lgbm_model():
    if LGBMRegressor is None:
        raise ImportError('lightgbm 패키지가 설치되어 있어야 합니다.')
    return LGBMRegressor(
        objective='regression',
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )


## 8. 모델 학습(Model Training, 모델 학습)

필수 순서:

1. `LinearRegression(선형회귀 모델)`
2. `LightGBM_RMSE(RMSE 기준 LightGBM)`

validation(검증) 결과를 기준으로 우세 모델을 선택한다.


In [8]:
results = []

X_train = train_2023[feature_cols]
y_train = train_2023[target_col]
X_valid = valid_2024[feature_cols]
y_valid = valid_2024[target_col]

linear_model = build_linear_pipeline()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_valid)
linear_metrics = evaluate_regression(y_valid, linear_pred)
results.append({
    'station_group': TARGET_STATION_GROUP,
    'model': 'LinearRegression',
    'split': 'validation_2024',
    **linear_metrics,
})

if LGBMRegressor is not None:
    lgbm_model = build_lgbm_model()
    lgbm_model.fit(X_train, y_train)
    lgbm_pred = lgbm_model.predict(X_valid)
    lgbm_metrics = evaluate_regression(y_valid, lgbm_pred)
    results.append({
        'station_group': TARGET_STATION_GROUP,
        'model': 'LightGBM_RMSE',
        'split': 'validation_2024',
        **lgbm_metrics,
    })

validation_results = pd.DataFrame(results)
validation_results


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003210 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1710
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 17
[LightGBM] [Info] Start training from score 1.423630


,station_group,model,split,rmse,mae,r2
0,아침 도착 업무 집중형,LinearRegression,validation_2024,1.761183,1.058876,0.562678
1,아침 도착 업무 집중형,LightGBM_RMSE,validation_2024,1.566030,0.926683,0.654226


## 9. 우세 모델 선택 및 재학습(Refit Best Model, 우세 모델 재학습)

- validation(검증) 결과를 기준으로 우세 모델을 선택한다.
- 선택 모델을 `2023 + 2024`로 재학습한다.
- 그 다음 `2025` 테스트셋 점수를 산출한다.


In [9]:
best_model_name = validation_results.sort_values('rmse').iloc[0]['model']
print('best_model_name:', best_model_name)

refit_train = train_feat.copy()
X_refit = refit_train[feature_cols]
y_refit = refit_train[target_col]
X_test = test_2025[feature_cols]
y_test = test_2025[target_col]

if best_model_name == 'LinearRegression':
    best_model = build_linear_pipeline()
else:
    best_model = build_lgbm_model()

best_model.fit(X_refit, y_refit)
test_pred = best_model.predict(X_test)
test_metrics = evaluate_regression(y_test, test_pred)

test_results = pd.DataFrame([{ 
    'station_group': TARGET_STATION_GROUP,
    'model': best_model_name,
    'split': 'test_2025_refit',
    **test_metrics,
}])
test_results


best_model_name: LightGBM_RMSE
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001460 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1748
[LightGBM] [Info] Number of data points in the train set: 52632, number of used features: 17
[LightGBM] [Info] Start training from score 1.444178


,station_group,model,split,rmse,mae,r2
0,아침 도착 업무 집중형,LightGBM_RMSE,test_2025_refit,1.346218,0.804153,0.639801


## 10. 결과 저장(Result Save, 결과 저장)

- 결과 CSV는 담당 군집명이 보이게 저장한다.
- 최소 2개 split(데이터 구간)인 `validation_2024(2024 검증 점수)`, `test_2025_refit(2025 재학습 후 최종 점수)`이 모두 있어야 한다.


In [10]:
final_metrics = pd.concat([validation_results, test_results], ignore_index=True)
safe_group_name = TARGET_STATION_GROUP.replace('/', '_').replace(' ', '_')
metrics_path = OUTPUT_DIR / f'ddri_{safe_group_name}_model_metrics.csv'
final_metrics.to_csv(metrics_path, index=False)
print('saved:', metrics_path)
final_metrics


saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/team_cluster_template_outputs/ddri_아침_도착_업무_집중형_model_metrics.csv


,station_group,model,split,rmse,mae,r2
0,아침 도착 업무 집중형,LinearRegression,validation_2024,1.761183,1.058876,0.562678
1,아침 도착 업무 집중형,LightGBM_RMSE,validation_2024,1.566030,0.926683,0.654226
2,아침 도착 업무 집중형,LightGBM_RMSE,test_2025_refit,1.346218,0.804153,0.639801


## 11. 결과 해석(Result Interpretation, 결과 해석)

아래 항목을 팀원별로 반드시 직접 작성한다.

- validation(검증) 기준 우세 모델은 무엇인가
- test(최종 평가) 점수는 어느 정도인가
- 이 군집에서 예측이 어려운 시간대가 보이는가
- 날씨나 1주 전 동일 시각(`lag_168h(168시간 전 대여량)`) 영향이 큰 것처럼 보이는가
- 다음 실험에서 추가할 피처(feature, 입력 변수) 또는 제거할 피처(feature, 입력 변수)는 무엇인가


## 12. 제출 전 체크리스트(Submission Checklist, 제출 전 점검표)

- [ ] 담당 군집 3개 대여소만 사용했는가
- [ ] `2023 train(학습) / 2024 validation(검증) / 2025 test(최종 평가)`를 지켰는가
- [ ] `LinearRegression(선형회귀 모델)`과 `LightGBM_RMSE(RMSE 기준 LightGBM)`를 모두 돌렸는가
- [ ] `RMSE(제곱평균제곱근오차)`, `MAE(평균절대오차)`, `R²(설명력)`를 validation/test 모두 기록했는가
- [ ] lag(과거 시점 값)/rolling(이동 통계)을 `station_id(숫자 대여소 ID)`별로 만들었는가
- [ ] 테스트셋으로 모델 선택을 하지 않았는가
- [ ] 노트북 마크다운에 전처리와 해석이 남아 있는가
- [ ] 차트와 표기가 `한글(영문)` 형식인가
